# Ejercicio Módulo 5 - Dataset Swiss
**Inteligencia Artificial - CEIA - FIUBA**

**Juan Ignacio Teich**

Para aprender sobre regresión, vamos a utilizar un dataset clásico llamado Swiss, que proviene originalmente del lenguaje R. Este dataset contiene datos socioeconómicos de 47 provincias suizas a fines del siglo XIX. Cada fila representa una provincia, y las variables reflejan características demográficas y sociales relevantes para ese contexto histórico.

## Variables

- `Location`: Provincia donde se midieron los datos.
- `Fertility`: Tasa de fertilidad (número promedio de hijos por mujer)
- `Agriculture`:` Porcentaje de hombres ocupados en agricultura
- `Examination`: Porcentaje de hombres que completaron exámenes de educación superior
- `Education`: Nivel promedio de educación (escala arbitraria)
- `Catholic`: Porcentaje de población católica
- `Infant.Mortality`: Tasa de mortalidad infantil (por cada 1000 nacidos vivos)

## Que queremos predecir?

Vamos a utilizar este dataset para predecir la tasa de fertilidad en cada provincia mediante diferentes métodos de regresión.

--- 

Siguiendo el procedimiento típico de Machine Learning, vamos a leer los datos y separarlos en los datasets de entrenamiento y testeo utilizando Scikit-Learn...

In [156]:
import pandas as pd

df = pd.read_csv("swiss.csv")

df.head()

,Location,Fertility,Agriculture,Examination,Education,Catholic,Infant.Mortality
0,Courtelary,80.2,17.0,15,12,9.96,22.2
1,Delemont,83.1,45.1,6,9,84.84,22.2
2,Franches-Mnt,92.5,39.7,5,5,93.40,20.2
3,Moutier,85.8,36.5,12,7,33.77,20.3
4,Neuveville,76.9,43.5,17,15,5.16,20.6


In [157]:
print(f"Tenemos {df.shape[0]} observaciones")

Tenemos 47 observaciones


Obtenemos la variable objetivo (`Fertility`) y, por otro lado, los atributos (quitamos `Location` ya que no es un atributo numérico relevante para la regresión)

In [158]:
X = df.drop(["Fertility", "Location"], axis=1)
y = df["Fertility"]

Dado que tenemos pocas observaciones, vamos a separar el dataset en un 50% para entrenamiento y 50% para testeo:

In [159]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

## Regresión lineal múltiple

Arranquemos la primera parte del ejercicio. Para eso, vamos a entrenar un modelo de regresión lineal múltiple usando todos los atributos. Para ello debes:

1. Escalar los atributos usando `StandardScaler`
2. Entrenar el modelo usando el dataset de entrenamiento.
3. Obtener las predicciones sobre el dataset de testeo.
4. Calcular las métricas MAE, MSE  y $R^2$, e imprimir los resultados.

In [160]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

##### COMPLETAR AQUI LO PEDIDO
# Punto 1: escalado de los atributos
# Lo hago directamente dentro de un pipeline
# Armemos el pipeline con StandardScaler para normalizar los datos, y LinearRegression para la regresión
pipeline = Pipeline(steps = [
    ('preprocessor', StandardScaler()),
    ('regressor', LinearRegression())
])

# Punto 2: entrenamiento del modelo
pipeline.fit(X_train, y_train)

# Veamos los valores de la función obtenida
# print(f"El valor de la intersección de la recta es {np.round(pipeline.named_steps['regressor'].intercept_, 2)}")
# print(f"Los valores de los coeficientes de la recta son {np.round(pipeline.named_steps['regressor'].coef_, 2)}")
# pipeline.named_steps['preprocessor'].get_feature_names_out()
# Veamos algunas métricas
# print(f"El coeficiente de determinación (R^2) en entrenamiento es {round(pipeline.score(X_train, y_train),3)}")

# Punto 3: predicción sobre el set de testeo
y_hat = pipeline.predict(X_test)

# Punto 4: Computemos las métricas pedidas
print("Punto 4:")
print("R^2 en test:", round(r2_score(y_test, y_hat),3))
print("MAE", round(mean_absolute_error(y_test, y_hat),3))
print("MSE", round(mean_squared_error(y_test, y_hat),3))

Punto 4:
R^2 en test: 0.574
MAE 6.094
MSE 64.457


## Modelo con regularización

Para mejorar nuestro modelo, vamos a explorar técnicas de regresión lineal con regularización, que nos permiten controlar el sobreajuste y seleccionar variables relevantes automáticamente.

Existen dos variantes muy populares:

- Una penaliza la suma de los cuadrados de los coeficientes (regularización L2).
- La otra penaliza la suma del valor absoluto de los coeficientes (regularización L1).

Ambas ayudan a mejorar la generalización, pero una de ellas además puede eliminar variables (coeficientes exactamente cero), lo que ayuda a identificar qué atributos son realmente importantes.

Tu tarea:

1. Elegí correctamente cuál de los dos métodos de regularización usar para este problema. 
    - Pista: Queremos que el modelo sea capaz de hacer una selección automática de variables, dejando fuera aquellas que no aportan.
2. Implementá un pipeline que incluya escalado y el modelo elegido.
3. Buscá automáticamente el mejor valor del hiperparámetro de regularización (alpha) usando validación cruzada usando 3-folds.
4. Entrená el modelo con los datos de entrenamiento y obtené las predicciones para el set de testeo.
5. Calcular las métricas MAE, MSE  y $R^2$, e imprimir los resultados.
6. Imprimí los coeficientes resultantes e identificá qué variables fueron eliminadas (coeficiente = 0).

In [161]:
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV

Paremos un momento para entender qué hacen LassoCV y RidgeCV antes de continuar con la resolución:

> Tanto `LassoCV` como `RidgeCV` son implementaciones de regresión lineal con regularización que incluyen la búsqueda automática del mejor hiperparámetro alpha mediante validación cruzada.
>
> Ambos métodos prueban distintos valores de alpha y eligen el que minimiza el error del modelo, facilitando el proceso de ajuste sin necesidad de una búsqueda manual.
>
> Internamente, utilizan la métrica del error cuadrático medio (MSE) para evaluar el rendimiento del modelo en cada fold de la validación cruzada.
>
> Por ejemplo, si llamás a RidgeCV(alphas=alphas, cv=5), se hará una validación cruzada de 5 folds utilizando los valores de alpha que vos le pases, y se seleccionará el que obtenga el menor MSE promedio.
> 
> Una vez elegido el mejor alpha, el modelo final se entrena con todos los datos de entrenamiento usando ese valor.

¡Listo! Con todo lo que vimos hasta ahora, ya estás en condiciones de resolver esta parte y completar los 6 puntos propuestos

**RESPUESTAS**:<br>
1. Como queremos sacar directamente las variables que no aportan, y no disminuir su influencia, vamos a usar la regularización L1 o de Lasso, ya que hace exactamente 0 el coeficiente de las variables menos importantes.

In [162]:
alphas = np.logspace(-4, 1, 500)

##### COMPLETAR AQUI LO PEDIDO
# Punto 2: implementar el pipeline
# Creemos el modelo
lasso_cv_pipeline = Pipeline(steps = [
    ('preprocessor', StandardScaler()),
    ('regressor', LassoCV(alphas=alphas, cv=3, random_state=33))
])

# Punto 3: hallar el alpha idóneo (y entrenar el modelo)
# Entrenemos el modelo
lasso_cv_pipeline.fit(X_train, y_train)
# Veamos su alpha
alpha = lasso_cv_pipeline.named_steps['regressor'].alpha_
print("Punto 3:")
print(f"El alpha de Lasso con validación cruzada 3-fold es {alpha}")

# Punto 4: predecir para el set de testeo
y_hat = lasso_cv_pipeline.predict(X_test)

# Punto 5: Calculemos las métricas
print("\nPunto 5:")
print("R^2 en test:", round(r2_score(y_test, y_hat),3))
print("MAE", round(mean_absolute_error(y_test, y_hat),3))
print("MSE", round(mean_squared_error(y_test, y_hat),3))

# Punto 6: Veamos los coeficientes
coeffs = lasso_cv_pipeline.named_steps['regressor'].coef_
print("\nPunto 6:")
print(f"Los coeficientes son: {coeffs}")
print(f"Recordemos las variables: {list(X_test.columns)}")



Punto 3:
El alpha de Lasso con validación cruzada 3-fold es 2.18110892419152

Punto 5:
R^2 en test: 0.576
MAE 5.995
MSE 64.294

Punto 6:
Los coeficientes son: [-0.         -1.76243263 -2.40490477  1.71964875  3.20040651]
Recordemos las variables: ['Agriculture', 'Examination', 'Education', 'Catholic', 'Infant.Mortality']


Sólo se eliminó la variable 'Agriculture'.

## Comparación de modelos y conclusiones

Completá la siguiente tabla con las métricas obtenidas para cada uno de los modelos que entrenaste:

| Modelo                        | MAE | MSE | $R^2$ |
| ----------------------------- | --- | --- | ----- |
| Regresión Lineal              | 6.094 | 64.457 | 0.574 |
| Modelo Regularizado (L1)      | 5.995 | 64.294 | 0.576 |


> ⚠️ Asegurate de cambiar el nombre del modelo `Modelo Regularizado (L1 o L2)` según el modelo que usaste (Lasso o Ridge).

### Justificación

**¿Cuál de los modelos te parece que tuvo un mejor desempeño general?**

Tené en cuenta las tres métricas al responder, y también pensá en la complejidad del modelo (por ejemplo, si eliminó variables innecesarias).

Escribí tu respuesta a continuación:

En cuanto a las métricas, ambos modelos se comportan de manera casi idéntica, la diferencia en $R^2$ es del 0.2%, del 1.6% en el MAE y del 0.2% en el MSE.<br>
El modelo regularizado tiene la ventaja de ser más sencillo al haber eliminado la variable 'Agriculture' que no aportaba información para la predicción de la fertilidad. Por este motivo podríamos decir que es el de mejor desempeño, aunque no de manera clara por performance.<br> 
Si el modelo regularizado que diera peores métricas fuera este, pero con un problema en el cual el descartar una variable resultará importante para reducir el tiempo/costo de predicción, también tendría sentido seguir eligiendolo a pesar de ser marginalmente peor en cuanto a métricas.